In [ ]:
!pip -q install unsloth
!pip -q uninstall unsloth -y && pip install --upgrade --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git@nightly git+https://github.com/unslothai/unsloth-zoo.git

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.7/69.7 kB 1.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 432.3/432.3 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 57.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 26.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 25.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 376.5/376.5 kB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 72.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 65.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 181.9/181.9 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
try:
  from google.colab import userdata
  from huggingface_hub import login
  HF_TOKEN = userdata.get('hf_token')
  login(HF_TOKEN)
  print('loggin successfuly')
except Exception as e:
  print(f'Error: {str(e)}')


loggin successfuly


In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

from unsloth import FastLanguageModel
import torch
import datasets
from trl import SFTTrainer
from transformers import TrainingArguments

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [ ]:
model_name = 'unsloth/Qwen2.5-1.5B-Instruct-unsloth-bnb-4bit'
repo_name = 'VyDat/Qwen3_translate'

data = 'VyDat/VSL-Dataset-10K'
max_seq_length = 128
dtype = None
SYS_PROMPT='Bạn là một chuyên gia ngôn ngữ, có thể tái cấu trúc câu từ ngôn ngữ nói Tiếng Việt thành cấu trúc câu Ngôn ngữ ký hiệu Việt Nam, hãy chuyển đổi câu sau:'

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    dtype=dtype,
    load_in_4bit=True,
    device_map="auto",
    max_seq_length=max_seq_length,
    )

==((====))==  Unsloth 2026.2.1: Fast Qwen3 patching. Transformers: 4.57.6.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/3.55G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/237 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=64,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=128,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing=True,
    random_state=42,
)


Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.2.1 patched 36 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


In [ ]:
def preprocess(example):
    messages = [
        {"role": "system", "content": SYS_PROMPT},
        {"role": "user", "content": example["source"]},
        {"role": "assistant", "content": example["target"]},
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False
    )

    tokenized = tokenizer(
        text,
        truncation=True,
        max_length=max_seq_length,
        padding=True
    )

    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

In [ ]:
data_vsl = datasets.load_dataset(data, split='train')
data_format = data_vsl.map(
    preprocess,
    remove_columns=data_vsl.column_names,
    desc="Running tokenizer on dataset",
    num_proc=4,
)

corpus_vie_vsl_10k.json: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Running tokenizer on dataset (num_proc=4):   0%|          | 0/10000 [00:00<?, ? examples/s]

In [ ]:
data_fromat = data_format.train_test_split(test_size=0.01, shuffle=True, seed=42)
data_train = data_fromat['train']
data_eval = data_format['test']

In [ ]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=data_train,
    eval_dataset=data_eval,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=4,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=8,
        gradient_accumulation_steps=4,
        warmup_steps=100,
        num_train_epochs=2,
        learning_rate=3e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=25,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=42,
        output_dir="outputs",
        save_strategy="steps",
        save_steps=100,
        eval_strategy="steps",
        eval_steps=200,
        save_total_limit=3,
        report_to="none",
    ),
)

In [ ]:
print("Start Trainning.....")
torch.cuda.empty_cache()

trainer_stats = trainer.train()

print(f"Trainning successfull, Total Loss: {trainer_stats.training_loss}")

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 9,900 | Num Epochs = 2 | Total steps = 620
O^O/ \_/ \    Batch size per device = 8 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (8 x 4 x 1) = 32
 "-____-"     Trainable parameters = 132,120,576 of 4,154,588,672 (3.18% trained)


Start Trainning.....


Step,Training Loss,Validation Loss
200,0.432100,0.417512
400,0.364200,0.400665
600,0.350000,0.391285


Unsloth: Not an error, but Qwen3ForCausalLM does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient


Trainning successfull, Total Loss: 0.46008205798364454


In [ ]:
print("Merging LoRA adapters into base model...")
model = model.merge_and_unload()

model.save_pretrained(repo_name)
tokenizer.save_pretrained(repo_name)

try:
    model.push_to_hub(repo_name, use_temp_dir=True)
    tokenizer.push_to_hub(repo_name, use_temp_dir=True)
    print(f"Training hoàn thành! Model đã được đẩy lên repo: https://huggingface.co/{repo_name}")
except Exception as e:
    print(f"Đã xảy ra lỗi khi đẩy model lên Hub: {e}")
    print(f"Model đã được lưu local tại './{repo_name}'")

Merging LoRA adapters into base model...


/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/bnb.py:397: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(


README.md:   0%|          | 0.00/581 [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

...

README.md:   0%|          | 0.00/587 [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mp5ocel637/tokenizer.json:  70%|######9   | 7.99MB / 11.4MB            

Training hoàn thành! Model đã được đẩy lên repo: https://huggingface.co/VyDat/Qwen3_translate
